# Combine All Category CSVs into One Dataset
Reads every `all - *.csv` from `data/raw/`, normalises the `name` and `wilayah` columns,
adds a `category` column, and saves to `data/processed/all_wilayah.csv`.

# Combine All Category CSVs into One Dataset
Reads every `all - *.csv` from `data/raw/`, normalises the `name` and `wilayah` columns,
adds a `category` column, and saves to `data/processed/all_wilayah.csv`.

In [1]:
import pandas as pd
from pathlib import Path

# ── Resolve project root ────────────────────────────────────────────────────
_cwd = Path.cwd()
BASE_DIR = _cwd
while not (BASE_DIR / 'data').exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent
print(f'Project root: {BASE_DIR.resolve()}')

RAW_DIR     = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR  = BASE_DIR / 'data' / 'processed'
OUTPUT_FILE = OUTPUT_DIR / 'all_wilayah.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Per-file config: (name_col, wilayah_col)  None = column not present ───
FILE_CONFIG = {
    'all - apart.csv':               ('Nama Apartemen',       'Wilayah Jakarta'),
    'all - bangunandanstruktur.csv':  ('Nama Gedung / Lokasi', 'Wilayah Jakarta'),
    'all - gedung_tinggi.csv':        ('Nama Gedung',          'Wilayah Jakarta'),
    'all - kompleks.csv':             ('Nama Komplek',         'Wilayah Jakarta'),
    'all - mall.csv':                 ('name',                 'Wilayah Jakarta'),
    'all - museum.csv':               ('Nama Museum',          'Kabupaten/Kota'),
    'all - rs.csv':                   ('nama_rumah_sakit',     'Unnamed: 1'),
    'all - sma_smk.csv':              ('Nama Sekolah',         'Wilayah Jakarta'),
    'all - taman.csv':                ('Nama Tempat',          'Wilayah Jakarta'),
    'all - tempat_wisata.csv':        ('nama_tempat_wisata',   'Unnamed: 1'),
    'all - univ.csv':                 ('nama_kampus',          None),
}

# ── Category label derived from filename ───────────────────────────────────
CATEGORY_MAP = {
    'all - apart.csv':               'apartemen',
    'all - bangunandanstruktur.csv':  'bangunan_dan_struktur',
    'all - gedung_tinggi.csv':        'gedung_tinggi',
    'all - kompleks.csv':             'kompleks',
    'all - mall.csv':                 'mall',
    'all - museum.csv':               'museum',
    'all - rs.csv':                   'rumah_sakit',
    'all - sma_smk.csv':              'sma_smk',
    'all - taman.csv':                'taman',
    'all - tempat_wisata.csv':        'tempat_wisata',
    'all - univ.csv':                 'universitas',
}

print('Config loaded.')

Project root: /Users/genevieve/College/thesis-address-normalization
Config loaded.


In [2]:
# ── Read & normalise each file ─────────────────────────────────────────────
frames = []

for filename, (name_col, wilayah_col) in FILE_CONFIG.items():
    path = RAW_DIR / filename
    if not path.exists():
        print(f'  SKIP (not found): {filename}')
        continue

    df = pd.read_csv(path)

    # Rename name column → 'name'
    df = df.rename(columns={name_col: 'name'})

    # Rename wilayah column → 'wilayah' (or add blank column if absent)
    if wilayah_col and wilayah_col in df.columns:
        df = df.rename(columns={wilayah_col: 'wilayah'})
    else:
        df['wilayah'] = None

    # Keep only the two standard columns
    df = df[['name', 'wilayah']].copy()

    # Add category
    df['category'] = CATEGORY_MAP[filename]

    # Basic cleaning
    df['name']    = df['name'].astype(str).str.strip()
    df['wilayah'] = df['wilayah'].astype(str).str.strip().replace('nan', None)
    df = df.dropna(subset=['name'])
    df = df[df['name'] != '']

    frames.append(df)
    print(f'  {filename:42s}  → {len(df):4d} rows')

combined = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(combined):,}')
combined.head(10)

  all - apart.csv                             →  363 rows
  all - bangunandanstruktur.csv               →  102 rows
  all - gedung_tinggi.csv                     →  118 rows
  all - kompleks.csv                          →  266 rows
  all - mall.csv                              →  107 rows
  all - museum.csv                            →   75 rows
  all - rs.csv                                →  196 rows
  all - sma_smk.csv                           → 1037 rows
  all - taman.csv                             →   37 rows
  all - tempat_wisata.csv                     →  151 rows
  all - univ.csv                              →  266 rows

Total rows: 2,718


,name,wilayah,category
0,Green Pramuka City,Jakarta Pusat,apartemen
1,Sudirman Park,Jakarta Pusat,apartemen
2,Semanggi,Jakarta Selatan,apartemen
3,Batavia,Jakarta Pusat,apartemen
4,Menteng Park,Jakarta Pusat,apartemen
5,Elpis Residence,Jakarta Pusat,apartemen
6,1 @ Cik Ditiro,Jakarta Pusat,apartemen
7,Kusuma Atmadja Residence,Jakarta Pusat,apartemen
8,Kimara Residence,Jakarta Pusat,apartemen
9,The Boulevard,Jakarta Pusat,apartemen


In [3]:
# ── Summary ────────────────────────────────────────────────────────────────
print('=== Rows per category ===')
print(combined['category'].value_counts().to_string())
print()
print('=== Rows per wilayah ===')
print(combined['wilayah'].value_counts().to_string())

=== Rows per category ===
category
sma_smk                  1037
apartemen                 363
kompleks                  266
universitas               266
rumah_sakit               196
tempat_wisata             151
gedung_tinggi             118
mall                      107
bangunan_dan_struktur     102
museum                     75
taman                      37

=== Rows per wilayah ===
wilayah
Jakarta Selatan     690
Jakarta Timur       506
Jakarta Barat       476
Jakarta Pusat       418
Jakarta Utara       338
None                266
Kepulauan Seribu     24


In [4]:
# ── Save ───────────────────────────────────────────────────────────────────
combined.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved {len(combined):,} rows → {OUTPUT_FILE.resolve()}')

Saved 2,718 rows → /Users/genevieve/College/thesis-address-normalization/data/processed/all_wilayah.csv
